# Lateral–Torsional Buckling — simply-supported I-beam, uniform moment

Same *geometric-nonlinearity-only* recipe as the chevron `buckleUP`
notebooks, now applied to the classical **lateral-torsional buckling**
(LTB) problem of a slender I-section beam in pure bending:

1. Build a single straight beam as one `add_line` call
2. Retrofit a **half-sine lateral imperfection** with `replace_line`
   (amplitude `L/1000`, direction = weak axis = global Y)
3. Emit the beam as 3-D `elasticBeamColumn` elements with a
   **`Corotational`** transformation — no material nonlinearity
4. Apply equal-and-opposite end moments so the beam sees a **uniform**
   strong-axis moment `M` over its full length
5. Sweep `M` in load control until the lateral deflection and twist
   amplify dramatically approaching the critical moment

### Why this captures LTB without modelling plasticity

LTB is a *geometric* instability: under strong-axis bending the
compression flange tends to drift laterally, which generates a
twisting moment on the cross-section, which amplifies the lateral
drift, which amplifies the twist… a classic coupled Euler-type mode.
Corotational finite rotations already model that coupling at the
element level — we do not need plasticity or any special LTB element.
The only ingredient the elastic formulation cannot supply by itself
is the **seed**: a perfectly straight beam stays perfectly straight
forever. A tiny lateral imperfection is that seed.

### Classical reference

For a simply-supported beam in pure uniform moment (no warping),

$$
M_{\mathrm{cr}} \;=\; \frac{\pi}{L_b}\,\sqrt{E\,I_{\text{weak}} \; G\,J}
$$

where `L_b` is the unbraced length, `I_weak` is the cross-section
moment of inertia about the **weak** axis, and `GJ` is the
St.-Venant torsional rigidity. The full Timoshenko formula adds a
`(π²/L²) · EI_weak · EC_w` warping term under the root — for the
W-section here that correction is ~15% and the FEM (which does **not**
carry warping stiffness in `elasticBeamColumn`) will track the
warping-free version cleanly.

In [2]:
from apeGmsh import apeGmsh, Results
import numpy as np
import matplotlib.pyplot as plt

# ---- Steel material ----------------------------------------------
E   = 200_000.0              # MPa
nu  = 0.3
G   = E / (2.0 * (1 + nu))

# ---- W-section geometry [mm] -------------------------------------
bf      = 150.0    # flange width
tf      = 12.0     # flange thickness
h_total = 300.0    # overall depth (web + 2 flanges)
tw      = 8.0      # web thickness
h_web   = h_total - 2 * tf

# ---- Derived section properties ----------------------------------
A  = 2 * bf * tf + h_web * tw
# Strong axis (local y in OpenSees): bending in the vertical plane.
I_strong = (2 * (bf * tf ** 3 / 12.0
                 + bf * tf * ((h_web + tf) / 2.0) ** 2)
            + tw * h_web ** 3 / 12.0)
# Weak axis (local z in OpenSees): bending in the horizontal plane.
I_weak   = 2 * (tf * bf ** 3 / 12.0) + h_web * tw ** 3 / 12.0
# Thin-walled open-section St.-Venant torsion constant.
J = (2 * bf * tf ** 3 + h_web * tw ** 3) / 3.0

# ---- Beam geometry -----------------------------------------------
L  = 6000.0     # unbraced length [mm]
lc = 150.0      # mesh size
delta_0 = L / 1000.0    # lateral imperfection amplitude

# ---- Classical LTB critical moment (no warping) ------------------
M_cr = (np.pi / L) * np.sqrt(E * I_weak * G * J)

print(f'Section W{h_total:.0f}x{int(bf)}x{int(tf)}')
print(f'  A        = {A:.0f} mm^2')
print(f'  I_strong = {I_strong:.3e} mm^4  (bending about strong axis)')
print(f'  I_weak   = {I_weak:.3e} mm^4    (bending about weak axis)')
print(f'  J        = {J:.0f} mm^4')
print(f'  I_strong / I_weak = {I_strong/I_weak:.1f}  '
      f'(larger is more LTB-prone)')
print()
print(f'Unbraced length L_b = {L:.0f} mm, imperfection L/1000 = {delta_0:.2f} mm')
print(f'M_cr (no warping)   = {M_cr/1e6:.2f} kN*m')

Section W300x150x12
  A        = 5808 mm^2
  I_strong = 8.871e+07 mm^4  (bending about strong axis)
  I_weak   = 6.762e+06 mm^4    (bending about weak axis)
  J        = 219904 mm^4
  I_strong / I_weak = 13.1  (larger is more LTB-prone)

Unbraced length L_b = 6000 mm, imperfection L/1000 = 6.00 mm
M_cr (no warping)   = 79.19 kN*m


## 1. Build the beam, then retrofit a half-sine lateral imperfection

The beam runs along the global X axis with the cross-section's
**strong axis vertical** (global Z). The weak axis — the one LTB
bows about — is therefore global Y. `replace_line` with
`direction=(0, 1, 0)` auto-projects that hint perpendicular to the
beam axis and places 15 interior points on a half-sine envelope of
amplitude `δ₀ = L/1000`.

In [3]:
m = apeGmsh(model_name='LTB_uniform', verbose=False)
m.begin()

p_pin  = m.model.geometry.add_point(0, 0, 0, lc=lc, label='pin')
p_roll = m.model.geometry.add_point(L, 0, 0, lc=lc, label='roller')

# Plain clean build first.
beam = m.model.geometry.add_line(p_pin, p_roll, label='beam')
m.physical.add_curve(tags=[beam], name='pg_beam')

# Retrofit with a half-sine imperfection in the weak-axis direction.
seg_tags = m.model.geometry.replace_line(
    beam,
    magnitude=delta_0,
    direction=(0, 1, 0),       # weak-axis lateral (global Y)
    shape='sine',
    n_segments=16,
)
print(f'beam replaced with {len(seg_tags)} sine segments: {seg_tags}')

m.mesh.sizing.set_global_size(lc)
m.mesh.generation.generate(dim=1)

fem = m.mesh.queries.get_fem_data(remove_orphans=True)
m.end()

print(f'nodes: {len(fem.nodes.ids)}')
for g in fem.elements:
    print(f'  {g.type_name:6s} n={len(g)}')

C:\Users\nmb\AppData\Local\Temp\ipykernel_27752\2280009032.py:1: UnitNotDeclaredWarning: apeGmsh(...): no consistent unit system declared (pass units='kN_m_s' or a baseUnits.systems module such as units=kN_m_s); Results and apeGmshViewer will require manual unit selection for unit conversions.
  m = apeGmsh(model_name='LTB_uniform', verbose=False)


beam replaced with 16 sine segments: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
nodes: 49
  line2  n=48
  point1 n=17


## 2. Inspect the imperfection

Walk every beam mesh node from pin to roller and compute the
perpendicular offset to the straight `(0,0,0) → (L,0,0)` axis.
Overlay the analytical half-sine envelope `δ₀·sin(π·x/L)` as a sanity
check.

In [4]:
beam_ids    = np.array(
    [int(n) for n in fem.nodes.select(target='pg_beam').ids])
beam_coords = fem.nodes.select(target='pg_beam').coords

order = np.argsort(beam_coords[:, 0])
beam_ids    = beam_ids[order]
beam_coords = beam_coords[order]

xs  = beam_coords[:, 0]
uy0 = beam_coords[:, 1]    # initial lateral offset (= the imperfection)
uz0 = beam_coords[:, 2]

x_theory = np.linspace(0, L, 300)
y_theory = delta_0 * np.sin(np.pi * x_theory / L)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(xs, uy0, 'o', ms=4, color='tab:blue', label='FEM node Y offsets')
ax.plot(x_theory, y_theory, 'k--', lw=1.2,
        label=f'\u03b4\u2080\u00b7sin(\u03c0\u00b7x/L), \u03b4\u2080={delta_0:.2f} mm')
ax.set_xlabel('x along beam [mm]')
ax.set_ylabel('initial y offset [mm]')
ax.set_title('Lateral imperfection shape')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'max FEM y offset = {uy0.max():.4f} mm  (theory = {delta_0:.4f} mm)')
print(f'max FEM z offset = {abs(uz0).max():.4f} mm  (should be 0)')

max FEM y offset = 6.0000 mm  (theory = 6.0000 mm)
max FEM z offset = 0.0000 mm  (should be 0)


C:\Users\nmb\AppData\Local\Temp\ipykernel_27752\4123542249.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Emit the 3-D OpenSees model

**`ndm=3, ndf=6`** for the rotational DOFs that LTB requires.

* **Cross-section orientation**: `geomTransf('Corotational', 1, 0, 0, 1)`
  places the local z axis along global Z (vertical) and the local y
  axis along global Y (horizontal/weak). The `elasticBeamColumn`
  argument order `(A, E, G, J, Iy, Iz, transf)` then takes `Iy = I_strong`
  (bending about local y = strong-axis bending) and `Iz = I_weak`.
* **Simply-supported fork support**: both ends have all three
  translations locked *except* the roller's axial DOF (free to let
  the beam stretch/shorten under moment-induced fibre strains), and
  both ends have the torsional DOF (rx) locked. The bending rotations
  (ry, rz) are left free so the beam can curve.
* **No material nonlinearity** — `elasticBeamColumn` is linear-elastic
  in the cross-section. The buckling comes entirely from the
  corotational finite-rotation kinematics.

In [5]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)
ops.timeSeries('Linear', 1)

for nid, xyz in fem.nodes.select():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

pin_id  = int(fem.nodes.select(target='pin').ids[0])
roll_id = int(fem.nodes.select(target='roller').ids[0])

# Fork supports:
#   pin    — ux, uy, uz fixed;         rx fixed; ry, rz free
#   roller — ux free; uy, uz fixed;    rx fixed; ry, rz free
ops.fix(pin_id,  1, 1, 1,  1, 0, 0)
ops.fix(roll_id, 0, 1, 1,  1, 0, 0)

# Local-z along global Z so bending about local-y = strong-axis.
ops.geomTransf('Corotational', 1, 0, 0, 1)

n_elem = 0
for group in fem.elements.select(element_type='line2').groups():
    for eid, conn in group:
        ops.element(
            'elasticBeamColumn', int(eid),
            int(conn[0]), int(conn[1]),
            A, E, G, J, I_strong, I_weak, 1,
        )
        n_elem += 1

print(f'elasticBeamColumn elements: {n_elem}')

elasticBeamColumn elements: 48


## 4. Apply equal-and-opposite end moments and sweep

For a **uniform** moment along the span we apply `+My` at the pin
and `−My` at the roller. Equilibrium then gives `M(x) = M_applied`
everywhere between them. The integrator advances the load factor so
the applied moment is just `(step+1) * dM`.

As `M` approaches the critical moment the lateral deflection and the
twist at midspan should both amplify sharply — classical LTB.

In [6]:
# Reference load = unit moment (Nm in each pattern).
ops.pattern('Plain', 1, 1)
ops.load(pin_id,  0, 0, 0,  0, +1.0, 0)    # +My at pin
ops.load(roll_id, 0, 0, 0,  0, -1.0, 0)    # -My at roller

ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-7, 200)
ops.algorithm('Newton')

M_target = 0.95 * M_cr
n_steps  = 50
dM       = M_target / n_steps
ops.integrator('LoadControl', dM)
ops.analysis('Static')

# Midspan tracking node (closest to x = L/2 in the original mesh).
mid_idx = int(np.argmin(np.abs(xs - L / 2.0)))
mid_nid = int(beam_ids[mid_idx])
print(f'midspan node = {mid_nid} at x = {xs[mid_idx]:.1f} mm')

hist_M       = []
hist_uy      = []
hist_uz      = []
hist_twist   = []
disp_per_step: list[np.ndarray] = []
twist_per_step: list[np.ndarray] = []
n_nodes = len(fem.nodes.ids)

for step in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f'analysis failed at step {step + 1}')
        break
    M_now = (step + 1) * dM
    d = ops.nodeDisp(mid_nid)
    # d = [ux, uy, uz, rx, ry, rz] — rx is the longitudinal-axis twist.
    hist_M.append(M_now)
    hist_uy.append(d[1])
    hist_uz.append(d[2])
    hist_twist.append(d[3])

    # Capture the full 3-D displacement + per-node twist for the viewer.
    d_full = np.zeros((n_nodes, 3), dtype=np.float64)
    t_full = np.zeros(n_nodes, dtype=np.float64)
    for i, nid in enumerate(fem.nodes.ids):
        di = ops.nodeDisp(int(nid))
        d_full[i, 0] = di[0]
        d_full[i, 1] = di[1]
        d_full[i, 2] = di[2]
        t_full[i]    = di[3]          # rx = twist about beam axis
    disp_per_step.append(d_full)
    twist_per_step.append(t_full)

print(f'converged in {len(hist_M)} of {n_steps} steps')
print(f'final M       = {hist_M[-1]/1e6:.3f} kN*m '
      f'({hist_M[-1]/M_cr*100:.1f}% M_cr)')
print(f'final uy mid  = {hist_uy[-1]:+.3f} mm  '
      f'(amplification \u03b4\u2080 \u2192 uy \u2248 '
      f'{(delta_0 + hist_uy[-1])/delta_0:.2f}x)')
print(f'final uz mid  = {hist_uz[-1]:+.3f} mm  (strong-axis bending)')
print(f'final twist   = {hist_twist[-1]:+.4e} rad  '
      f'({np.degrees(hist_twist[-1]):+.2f} deg)')

midspan node = 10 at x = 3000.0 mm
converged in 50 of 50 steps
final M       = 75.234 kN*m (95.0% M_cr)
final uy mid  = +27.735 mm  (amplification δ₀ → uy ≈ 5.62x)
final uz mid  = -23.491 mm  (strong-axis bending)
final twist   = -1.5004e-01 rad  (-8.60 deg)


## 5. Amplification curves vs. Perry-like envelope

LTB is a *coupled* (lateral + twist) buckling mode, so the classical
Perry formula `δ(P) = δ₀/(1-P/P_cr)` is an **envelope** rather than an
exact match — but the asymptote at `M_cr` is the same. We plot:

* **M vs lateral uy** (Perry envelope `δ₀/(1−M/M_cr)`)
* **M vs twist** (from the FEM)
* **M vs strong-axis uz** (linear until twist kicks in)

In [7]:
hist_M     = np.asarray(hist_M)
hist_uy    = np.asarray(hist_uy)
hist_uz    = np.asarray(hist_uz)
hist_twist = np.asarray(hist_twist)

# Perry envelope using theoretical M_cr.
eta = hist_M / M_cr
valid = eta < 0.99
uy_theory = delta_0 / (1.0 - eta[valid])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

# (a) Lateral deflection
ax = axes[0]
ax.plot(delta_0 + hist_uy, hist_M / 1e6, 'o-', lw=1.4, ms=3,
        color='tab:red', label='FEM (lateral)')
ax.plot(uy_theory, hist_M[valid] / 1e6, 'k--', lw=1.2,
        label='Perry envelope  \u03b4\u2080/(1-M/M_cr)')
ax.axhline(M_cr / 1e6, color='k', lw=0.8, ls=':',
           label=f'M_cr = {M_cr/1e6:.1f} kN*m')
ax.axvline(delta_0, color='tab:gray', lw=0.8, ls=':',
           label=f'\u03b4\u2080 = L/1000 = {delta_0:.2f} mm')
ax.set_xlabel('midspan lateral deflection uy [mm]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Lateral amplification vs. Perry envelope')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=9)

# (b) Twist
ax = axes[1]
ax.plot(np.degrees(hist_twist), hist_M / 1e6, 'o-', lw=1.4, ms=3,
        color='tab:purple')
ax.axhline(M_cr / 1e6, color='k', lw=0.8, ls=':')
ax.set_xlabel('midspan twist rx [deg]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Twist vs. moment')
ax.grid(True, alpha=0.3)

# (c) Strong-axis vertical deflection (should be roughly linear)
ax = axes[2]
ax.plot(hist_uz, hist_M / 1e6, 'o-', lw=1.4, ms=3, color='tab:blue',
        label='FEM (vertical)')
# Linear reference: M = EI_strong * kappa, kappa = 8*uz/L^2 for a
# uniform-moment simply-supported beam at midspan.
uz_linear_grid = np.linspace(hist_uz.min(), 0, 50)
M_linear_grid  = np.abs(uz_linear_grid) * 8 * E * I_strong / L**2
ax.plot(uz_linear_grid, M_linear_grid / 1e6, 'k--', lw=1.0,
        label='linear beam theory')
ax.axhline(M_cr / 1e6, color='k', lw=0.8, ls=':')
ax.set_xlabel('midspan vertical deflection uz [mm]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Strong-axis bending')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

C:\Users\nmb\AppData\Local\Temp\ipykernel_27752\2917132412.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Time-series viewer with twist contour

Push every load step into `Results.from_fem` as a pseudo-time frame.
Each step carries the displacement vector + three scalars:

* `|U|` — magnitude for contour colouring of the deformed shape
* `Uy` — weak-axis lateral displacement (the one that amplifies)
* `twist_deg` — `rx` converted to degrees, so you can see the
  compression flange progressively rolling out of the vertical plane

## Record results

The cell below re-runs the analysis and records it, choosing the route by
the OpenSees build the deck ran on:

- **Ladruno fork:** ``ResolvedRecorderSpec.emit_default`` records the run
  with the fork's own recorder (a ``.ladruno`` file).
- **Stock openseespy:** the cell samples the live domain after every step
  and writes a native results file with ``NativeWriter``. ``emit_default``
  cannot record this deck on a stock build without MPCO: its ``.out``
  route needs a bridge model archive that declares ``ndm``, and this deck
  was built by hand.

Either file is read back with ``Results.from_fem(fem, path)``: the deck
tags nodes and elements by their FEM ids, so the FEMData snapshot is the
model. The cell then checks the recorded last step against the live
domain. ``Results.viewer`` honours ``APEGMSH_SKIP_VIEWER``, so this
notebook is safe to run under nbconvert / CI without changes.


In [ ]:
# --- EOS-WIRING-V2 ---
# Record the run: the fork's own recorder on the Ladruno fork, a native
# file sampled from the live domain on stock openseespy.
import contextlib
import numpy as np
from apeGmsh import Results, workdir
from apeGmsh.opensees._target import backend_info_of
from apeGmsh.results.spec import ResolvedRecorderRecord, ResolvedRecorderSpec
from apeGmsh.results.writers import NativeWriter

OUT = workdir()

# --- copied from cell 7 ---
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)
ops.timeSeries('Linear', 1)

for nid, xyz in fem.nodes.select():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

pin_id  = int(fem.nodes.select(target='pin').ids[0])
roll_id = int(fem.nodes.select(target='roller').ids[0])

# Fork supports:
#   pin    — ux, uy, uz fixed;         rx fixed; ry, rz free
#   roller — ux free; uy, uz fixed;    rx fixed; ry, rz free
ops.fix(pin_id,  1, 1, 1,  1, 0, 0)
ops.fix(roll_id, 0, 1, 1,  1, 0, 0)

# Local-z along global Z so bending about local-y = strong-axis.
ops.geomTransf('Corotational', 1, 0, 0, 1)

n_elem = 0
for group in fem.elements.select(element_type='line2').groups():
    for eid, conn in group:
        ops.element(
            'elasticBeamColumn', int(eid),
            int(conn[0]), int(conn[1]),
            A, E, G, J, I_strong, I_weak, 1,
        )
        n_elem += 1

print(f'elasticBeamColumn elements: {n_elem}')
# --- copied from cell 9 ---
# Reference load = unit moment (Nm in each pattern).
ops.pattern('Plain', 1, 1)
ops.load(pin_id,  0, 0, 0,  0, +1.0, 0)    # +My at pin
ops.load(roll_id, 0, 0, 0,  0, -1.0, 0)    # -My at roller

ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-7, 200)
ops.algorithm('Newton')

M_target = 0.95 * M_cr
n_steps  = 50
dM       = M_target / n_steps
ops.integrator('LoadControl', dM)
ops.analysis('Static')

# Midspan tracking node (closest to x = L/2 in the original mesh).
mid_idx = int(np.argmin(np.abs(xs - L / 2.0)))
mid_nid = int(beam_ids[mid_idx])
print(f'midspan node = {mid_nid} at x = {xs[mid_idx]:.1f} mm')

hist_M       = []
hist_uy      = []
hist_uz      = []
hist_twist   = []
disp_per_step: list[np.ndarray] = []
twist_per_step: list[np.ndarray] = []
n_nodes = len(fem.nodes.ids)

# Record every FEM node the deck built.
_axes = ("x", "y", "z")[:3]
_built = set(int(n) for n in ops.getNodeTags())
_node_ids = np.asarray(
    [int(n) for n in fem.nodes.ids if int(n) in _built], dtype=np.int64)
_backend = backend_info_of(ops)
_fork = _backend.kind == "fork"
print(f"OpenSees build: {_backend.kind} ({_backend.source})")

if _fork:
    spec = ResolvedRecorderSpec(
        fem_snapshot_id=fem.snapshot_id,
        records=tuple(
            ResolvedRecorderRecord(
                category="nodes", name=kind,
                components=tuple(f"{kind}_{a}" for a in _axes),
                dt=None, n_steps=None, node_ids=_node_ids,
            )
            for kind in ("displacement", "reaction_force")
        ),
    )
    # emit_default needs a model archive on disk; the deck tags nodes and
    # elements by FEM id, so the FEMData archive is that model.
    model_h5 = OUT / "model.h5"
    fem.to_h5(str(model_h5))
    _recording = spec.emit_default(OUT / "run", model_h5=model_h5, ops=ops,
                                   overwrite=True)
else:
    _recording = contextlib.nullcontext()

_steps = []   # stock: (time, displacement) after every step


def _record_step():
    if not _fork:
        _steps.append((ops.getTime(), np.array(
            [ops.nodeDisp(int(n))[:3] for n in _node_ids])))


with _recording as run:
    for step in range(n_steps):
        ok = ops.analyze(1)
        _record_step()
        if ok != 0:
            print(f'analysis failed at step {step + 1}')
            break
        M_now = (step + 1) * dM
        d = ops.nodeDisp(mid_nid)
        # d = [ux, uy, uz, rx, ry, rz] — rx is the longitudinal-axis twist.
        hist_M.append(M_now)
        hist_uy.append(d[1])
        hist_uz.append(d[2])
        hist_twist.append(d[3])

        # Capture the full 3-D displacement + per-node twist for the viewer.
        d_full = np.zeros((n_nodes, 3), dtype=np.float64)
        t_full = np.zeros(n_nodes, dtype=np.float64)
        for i, nid in enumerate(fem.nodes.ids):
            di = ops.nodeDisp(int(nid))
            d_full[i, 0] = di[0]
            d_full[i, 1] = di[1]
            d_full[i, 2] = di[2]
            t_full[i]    = di[3]          # rx = twist about beam axis
        disp_per_step.append(d_full)
        twist_per_step.append(t_full)

    print(f'converged in {len(hist_M)} of {n_steps} steps')
    print(f'final M       = {hist_M[-1]/1e6:.3f} kN*m '
          f'({hist_M[-1]/M_cr*100:.1f}% M_cr)')
    print(f'final uy mid  = {hist_uy[-1]:+.3f} mm  '
          f'(amplification \u03b4\u2080 \u2192 uy \u2248 '
          f'{(delta_0 + hist_uy[-1])/delta_0:.2f}x)')
    print(f'final uz mid  = {hist_uz[-1]:+.3f} mm  (strong-axis bending)')
    print(f'final twist   = {hist_twist[-1]:+.4e} rad  '
          f'({np.degrees(hist_twist[-1]):+.2f} deg)')

if _fork:
    results_path = run.path
    results = Results.from_fem(fem, results_path, cache_root=OUT)
else:
    results_path = OUT / "run_native.h5"
    if results_path.exists():
        results_path.unlink()
    _u = np.stack([u for _, u in _steps])          # (T, N, ndm)
    with NativeWriter(results_path) as _nw:
        _nw.open(fem=fem, source_type="native", opensees_backend=_backend.kind)
        _sid = _nw.begin_stage(name="analysis", kind="static",
                               time=np.array([t for t, _ in _steps]))
        _nw.write_nodes(_sid, "partition_0", node_ids=_node_ids, components={
            f"displacement_{a}": _u[:, :, k] for k, a in enumerate(_axes)})
        _nw.end_stage()
    results = Results.from_fem(fem, results_path, kind="native", cache_root=OUT)
print(f"recorded -> {results_path}")

# The recorded last step is the live domain's displacement field.
_slabs = [results.nodes.get(ids=_node_ids, component=f"displacement_{a}")
          for a in _axes]
assert sorted(int(n) for n in _slabs[0].node_ids) == sorted(_node_ids.tolist())
_rec = np.column_stack([s.values[-1] for s in _slabs])
_live = np.array([ops.nodeDisp(int(n))[:3] for n in _slabs[0].node_ids])
_scale = max(float(np.abs(_live).max()), 1e-300)
assert np.allclose(_rec, _live, rtol=1e-6, atol=1e-9 * _scale), (
    "the recorded displacement disagrees with the live domain")
print(f"recorded last step matches the live domain at {len(_node_ids)} nodes")


In [ ]:
# Open the recorded run in the apeGmsh ResultsViewer (subprocess, non-blocking).
# Set APEGMSH_SKIP_VIEWER=1 in the environment to skip the GUI under nbconvert / CI.
results.viewer(blocking=False)
